In [6]:
import json
from time import time

import pandas as pd
from kafka import KafkaProducer

In [7]:
TOPIC = "green-trips"
DATA_URL = "https://github.com/DataTalksClub/nyc-tlc-data/releases/download/green/green_tripdata_2019-10.csv.gz"

COLUMNS = [
    "lpep_pickup_datetime",
    "lpep_dropoff_datetime",
    "PULocationID",
    "DOLocationID",
    "passenger_count",
    "trip_distance",
    "tip_amount",
]

In [8]:
def json_serializer(data):
    return json.dumps(data).encode("utf-8")

df = pd.read_csv(DATA_URL, compression="gzip", low_memory=False)
df = df[COLUMNS]
df = df.where(pd.notnull(df), None)
df.head()

,lpep_pickup_datetime,lpep_dropoff_datetime,PULocationID,DOLocationID,passenger_count,trip_distance,tip_amount
0,2019-10-01 00:26:02,2019-10-01 00:39:58,112,196,1.0,5.88,0.00
1,2019-10-01 00:18:11,2019-10-01 00:22:38,43,263,1.0,0.80,0.00
2,2019-10-01 00:09:31,2019-10-01 00:24:47,255,228,2.0,7.50,0.00
3,2019-10-01 00:37:40,2019-10-01 00:41:49,181,181,1.0,0.90,0.00
4,2019-10-01 00:08:13,2019-10-01 00:17:56,97,188,1.0,2.52,2.26


In [9]:
producer = KafkaProducer(
    bootstrap_servers=["localhost:9092"],
    value_serializer=json_serializer,
)

t0 = time()

for row in df.to_dict(orient="records"):
    producer.send(TOPIC, value=row)

producer.flush()

t1 = time()
print(f"Rows sent: {len(df)}")
print(f"Took: {t1 - t0:.2f} seconds")

ERROR! Session/line number was not unique in database. History logging moved to new session 16
Rows sent: 476386
Took: 18.21 seconds
